#تمرین سوم درس بینایی کامپیوتر
 در این سوال قصد داریم با داشتن تصاویر از زوایه‌های مختلف یک صحنه، یک بازسازی پانوراما از صحنه داشته باشیم

کتابخانه‌های لازم را در زیر اضافه کنید

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import signal as sig
import scipy.ndimage as ndi
from skimage.io import imread
from skimage.color import rgb2gray
from skimage import data
from skimage.feature import match_descriptors, ORB, plot_matches
from skimage.measure import ransac
from skimage.transform import FundamentalMatrixTransform
import skimage.feature as sf
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

### تولید تصویر پانوراما از دو تصویر
در این قسمت ابتدا دو تصویر مختف از یک صحنه را داریم و سعی میکنیم تصویر پانوراما این دو تصویر را تولید کنیم. اتبدا دو تصویر را خوانده و برای هر تصویر تعدادی نقطه کلیدی یا $key point$ استخراج می‌کنیم. سپس این نقاط کلیدی را در تصویر با همدیگر مچ می‌کنیم. درمورد روش‌های استخراج ویژگی و مچ کردن آنها در فصل بعد آشنا خواهید شد. در این قطعه کد از توابع آماده کتابخانه $OpenCV$ برای این کار استفاده می‌کنیم. ابتدا دوتصویر را نمایش نمایش می‌دهیم

In [ ]:
image1 = cv2.imread('')
image2 = cv2.imread('')



در قسمت بعد استخراج نقاط کلیدی و مچ کردن آنها اتفاق می‌افتد. نقاط مچ شده را در تصویر خروجی می‌بینید

In [ ]:
gray2= cv2.cvtColor(image1,cv2.COLOR_BGR2GRAY)
gray1= cv2.cvtColor(image2,cv2.COLOR_BGR2GRAY)

# key point extraction


در تابع زیر با گرفتن دو تصویر ورودی، ماتریس هموگرافی بین آنها محاسبه شده و نقاط مربوطه بر روی همدیگر نگاشت شده و تصویر پانوراما را در خروجی برمی‌گرداند. تایع دیگر نیز ماتریس هموگرافی بین دو تصویر را محاسبه می‌کند

In [ ]:
def findHomography(kp1, kp2, matches):
    img1_points = np.zeros((len(matches), 1, 2), dtype=np.float32)
    img2_points = np.zeros((len(matches), 1, 2), dtype=np.float32)

    for i in range(0,len(matches)):
        img1_points[i] = kp1[matches[i].queryIdx].pt
        img2_points[i] = kp2[matches[i].trainIdx].pt

    homography, mask = cv2.findHomography(img1_points, img2_points, cv2.RANSAC, ransacReprojThreshold=2.0)

    return homography

#Compute hom and homInv
hom = findHomography(kp1, kp2, matches)
homInv = np.linalg.inv(hom)

In [ ]:
def stitch(img1, img2, hom):
  rows1, cols1 = img1.shape[:2]
  rows2, cols2 = img2.shape[:2]

  #Corners of img1 & img2
  corners1 = np.float32([[0,0], [0, rows1],[cols1, rows1], [cols1, 0]]).reshape(-1, 1, 2)
  corners2 = np.float32([[0,0], [0,rows2], [cols2,rows2], [cols2,0]]).reshape(-1,1,2)
  
  #Project 4 corners of image2 onto image1
  project = cv2.perspectiveTransform(corners2, hom)

  points = np.concatenate((corners1, project), axis = 0)

  [x_min, y_min] = np.int32(points.min(axis = 0).ravel() - 0.5)
  [x_max, y_max] = np.int32(points.max(axis = 0).ravel() + 0.5)
  
  translation_dist = [-x_min,-y_min]
  
  H_translation = np.array([[1, 0, translation_dist[0]], [0, 1, translation_dist[1]], [0, 0, 1]])

  result = cv2.warpPerspective(img2, H_translation.dot(hom), (x_max - x_min, y_max - y_min))
  result[translation_dist[1] : rows1 + translation_dist[1], translation_dist[0] : cols1 + translation_dist[0]] = img1

  return result

نمونه تصویر پانوراما تولید شده

In [ ]:
panorama = stitch(image1, image2, homInv)
cv2_imshow(cv2.resize(panorama, (1300, 600), interpolation = cv2.INTER_AREA))

### تولید تصویر پانوراما از چندین تصویر
در این قسمت تمامی کارهای قسمت قبل را این بار برای چندین توصیر از یک صحنه انجام ‌می‌دهیم. این بار از توابع آماده کتابخانه $openCV$ استفاده می‌کنیم

In [ ]:
img1 = cv2.imread('1.jpg')
img2 = cv2.imread('2.jpg')
img3 = cv2.imread('3.jpg')
img4 = cv2.imread('4.jpg')
img5 = cv2.imread('5.jpg')
img6 = cv2.imread('6.jpg')



در نهایت تصویر پانورامای 6 تصویر بالا را محاسبه کرده و در خروجی نمایش می‌دهیم